In [1]:
import os

# RT-DETR-L T4x2 attempt with a Kaggle-only DDP patch. The previous 300-epoch
# single-T4 run was too slow for Kaggle's 12-hour limit. This run keeps the
# stronger RT-DETR-L model and 640px images, but uses both T4s and a wall-clock
# budget so submission generation and artifact packaging can run before Kaggle's limit.
os.environ["AMIA_MODEL_KIND"] = "rtdetr"
os.environ["AMIA_MODEL"] = "rtdetr-l.pt"
os.environ["AMIA_EPOCHS"] = "20"
os.environ["AMIA_IMGSZ"] = "640"
os.environ["AMIA_BATCH"] = "8"  # global DDP batch; each T4 receives roughly half
os.environ["AMIA_WORKERS"] = "4"
os.environ["AMIA_DEVICE"] = "0,1"
os.environ["AMIA_PATIENCE"] = "8"
os.environ["AMIA_TRAIN_TIME_HOURS"] = "10.5"
os.environ["AMIA_SAVE_PERIOD"] = "5"
os.environ["AMIA_DETERMINISTIC"] = "false"
os.environ["AMIA_DDP_FIND_UNUSED_PARAMETERS"] = "true"
os.environ["AMIA_RUN_NAME"] = "rtdetr-l-e20-b8-t4x2-ddpfix-t10h5"
os.environ["AMIA_PRED_CONF"] = "0.05"
os.environ["AMIA_PRED_IOU"] = "0.4"
os.environ["AMIA_PRED_MAX_DETECTIONS"] = "300"

In [2]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path


COMPETITION = "amia-public-challenge-2026"
KAGGLE_INPUT = Path(os.environ.get("AMIA_INPUT", f"/kaggle/input/{COMPETITION}"))
WORKING_DIR = Path(os.environ.get("AMIA_WORKING", "/kaggle/working"))
DETECTOR_DIR = WORKING_DIR / "amia-detector"
MODEL_DIR = WORKING_DIR / "models"
SPLIT_SEED = os.environ.get("AMIA_SPLIT_SEED", "cxr-amia-v1")
VAL_FRACTION = float(os.environ.get("AMIA_VAL_FRACTION", "0.2"))

EPOCHS = int(os.environ.get("AMIA_EPOCHS", "300"))
IMAGE_SIZE = int(os.environ.get("AMIA_IMGSZ", "640"))
BATCH = int(os.environ.get("AMIA_BATCH", "8"))
WORKERS = int(os.environ.get("AMIA_WORKERS", "4"))
MODEL_KIND = os.environ.get("AMIA_MODEL_KIND", "rtdetr").lower()
MODEL = os.environ.get("AMIA_MODEL", "rtdetr-l.pt")
RUN_NAME = os.environ.get("AMIA_RUN_NAME", f"{MODEL_KIND}-{EPOCHS}-b{BATCH}")
DEVICE = os.environ.get("AMIA_DEVICE", "auto")
PATIENCE = int(os.environ.get("AMIA_PATIENCE", "15"))
TRAIN_TIME_HOURS = float(os.environ.get("AMIA_TRAIN_TIME_HOURS", "0"))
SAVE_PERIOD = int(os.environ.get("AMIA_SAVE_PERIOD", "1"))
DETERMINISTIC = os.environ.get("AMIA_DETERMINISTIC", "true").lower() in {"1", "true", "yes", "on"}
DDP_FIND_UNUSED_PARAMETERS = os.environ.get("AMIA_DDP_FIND_UNUSED_PARAMETERS", "false").lower() in {"1", "true", "yes", "on"}
PRED_CONF = float(os.environ.get("AMIA_PRED_CONF", "0.05"))
PRED_IOU = float(os.environ.get("AMIA_PRED_IOU", "0.4"))
PRED_BATCH = int(os.environ.get("AMIA_PRED_BATCH", "8"))
PRED_CHUNK_SIZE = int(os.environ.get("AMIA_PRED_CHUNK_SIZE", "128"))
PRED_MAX_DETECTIONS = int(os.environ.get("AMIA_PRED_MAX_DETECTIONS", "300"))
PRED_DEVICE = os.environ.get("AMIA_PRED_DEVICE", "auto")

NO_FINDING_CLASS_ID = "14"
CLASS_NAMES = {
    "0": "Aortic enlargement",
    "1": "Atelectasis",
    "2": "Calcification",
    "3": "Cardiomegaly",
    "4": "Consolidation",
    "5": "ILD",
    "6": "Infiltration",
    "7": "Lung Opacity",
    "8": "Nodule/Mass",
    "9": "Other lesion",
    "10": "Pleural effusion",
    "11": "Pleural thickening",
    "12": "Pneumothorax",
    "13": "Pulmonary fibrosis",
}

In [3]:
def ensure_ultralytics() -> None:
    """Install Ultralytics in Kaggle if it is not already importable."""

    try:
        import ultralytics
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "ultralytics"]
        )
        import ultralytics
    print(f"ultralytics={ultralytics.__version__}")


def patch_ultralytics_ddp_find_unused_parameters() -> None:
    """Patch Kaggle's temporary Ultralytics install to tolerate RT-DETR DDP unused params."""

    if not DDP_FIND_UNUSED_PARAMETERS:
        print("Ultralytics DDP patch disabled")
        return

    import ultralytics

    trainer_path = Path(ultralytics.__file__).parent / "engine" / "trainer.py"
    text = trainer_path.read_text()
    if "find_unused_parameters=True" in text:
        print(f"Ultralytics DDP patch already present: {trainer_path}")
        return

    old = """self.model = nn.parallel.DistributedDataParallel(
                self.model,
                device_ids=[RANK],
                static_graph=bool(self.args.compile),
            )"""
    new = """self.model = nn.parallel.DistributedDataParallel(
                self.model,
                device_ids=[RANK],
                static_graph=bool(self.args.compile),
                find_unused_parameters=True,
            )"""
    if old not in text:
        raise RuntimeError(f"Could not find Ultralytics DDP block to patch in {trainer_path}")
    trainer_path.write_text(text.replace(old, new))
    print(f"Patched Ultralytics DDP with find_unused_parameters=True: {trainer_path}")


def find_competition_input(preferred: Path) -> Path:
    """Find the Kaggle input directory containing the competition CSV files."""

    required_files = ("train.csv", "img_size.csv", "sample_submission.csv")
    if all((preferred / name).exists() for name in required_files):
        return preferred

    input_root = Path("/kaggle/input")
    candidates = []
    if input_root.exists():
        for train_csv in sorted(input_root.rglob("train.csv")):
            candidate = train_csv.parent
            if all((candidate / name).exists() for name in required_files):
                candidates.append(candidate)

    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        formatted = "\n".join(f"- {candidate}" for candidate in candidates)
        raise FileNotFoundError(
            "Multiple possible competition input directories found. "
            "Set AMIA_INPUT explicitly before running training:\n"
            f"{formatted}"
        )

    visible = sorted(str(path) for path in input_root.glob("*")) if input_root.exists() else []
    raise FileNotFoundError(
        "Could not find competition files under /kaggle/input. "
        "Attach the AMIA Public Challenge 2026 competition data, or set "
        "AMIA_INPUT to the directory containing train.csv, img_size.csv, and "
        f"sample_submission.csv. Visible inputs: {visible}"
    )


def read_csv_dicts(path: Path) -> list[dict[str, str]]:
    """Read a CSV file into dictionaries."""

    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def deterministic_fraction(image_id: str, seed: str) -> float:
    """Map an image ID to a stable fraction in [0, 1)."""

    digest = hashlib.sha256(f"{seed}:{image_id}".encode("utf-8")).hexdigest()
    return int(digest[:16], 16) / float(16**16)


def yolo_label_line(row: dict[str, str], height: float, width: float) -> str:
    """Convert one absolute-pixel box row into YOLO normalized coordinates."""

    x_min = float(row["x_min"])
    y_min = float(row["y_min"])
    x_max = float(row["x_max"])
    y_max = float(row["y_max"])
    if not (0 <= x_min < x_max <= width and 0 <= y_min < y_max <= height):
        raise ValueError(
            f"Invalid box for {row['image_id']}: {(x_min, y_min, x_max, y_max)} "
            f"outside width={width}, height={height}"
        )
    x_center = ((x_min + x_max) / 2.0) / width
    y_center = ((y_min + y_max) / 2.0) / height
    box_width = (x_max - x_min) / width
    box_height = (y_max - y_min) / height
    return (
        f"{row['class_id']} {x_center:.8f} {y_center:.8f} "
        f"{box_width:.8f} {box_height:.8f}"
    )


def link_or_copy(source: Path, target: Path) -> None:
    """Symlink an image into the working dataset, falling back to copy."""

    if target.exists() or target.is_symlink():
        target.unlink()
    try:
        target.symlink_to(source)
    except OSError:
        shutil.copy2(source, target)


def prepare_detector_dataset(input_dir: Path, output_dir: Path) -> Path:
    """Build an Ultralytics detection dataset under `/kaggle/working`."""

    train_csv = input_dir / "train.csv"
    image_size_csv = input_dir / "img_size.csv"
    train_images = input_dir / "train" / "train"
    if output_dir.exists():
        shutil.rmtree(output_dir)

    for split in ("train", "val"):
        (output_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (output_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    rows = read_csv_dicts(train_csv)
    image_sizes = {
        row["image_id"]: (float(row["dim0"]), float(row["dim1"]))
        for row in read_csv_dicts(image_size_csv)
    }

    classes_by_image: dict[str, set[str]] = defaultdict(set)
    rows_by_image: dict[str, list[dict[str, str]]] = defaultdict(list)
    skipped_no_finding_rows = 0
    for row in rows:
        image_id = row["image_id"]
        classes_by_image[image_id].add(row["class_id"])
        if row["class_id"] == NO_FINDING_CLASS_ID:
            skipped_no_finding_rows += 1
        else:
            rows_by_image[image_id].append(row)

    counts = Counter()
    for image_id in sorted(classes_by_image):
        split = "val" if deterministic_fraction(image_id, SPLIT_SEED) < VAL_FRACTION else "train"
        if image_id not in image_sizes:
            raise ValueError(f"Missing image size for {image_id}")
        height, width = image_sizes[image_id]
        label_lines = [
            yolo_label_line(row, height, width)
            for row in sorted(rows_by_image.get(image_id, []), key=lambda item: item["class_id"])
        ]
        (output_dir / "labels" / split / f"{image_id}.txt").write_text(
            "\n".join(label_lines) + ("\n" if label_lines else "")
        )
        source_image = train_images / f"{image_id}.png"
        if not source_image.exists():
            raise FileNotFoundError(f"Missing training image: {source_image}")
        link_or_copy(source_image, output_dir / "images" / split / f"{image_id}.png")
        counts[f"{split}_images"] += 1
        counts[f"{split}_boxes"] += len(label_lines)
        if not label_lines:
            counts[f"{split}_backgrounds"] += 1

    names = "\n".join(f"  {class_id}: {CLASS_NAMES[str(class_id)]}" for class_id in range(14))
    dataset_yaml = output_dir / "dataset.yaml"
    dataset_yaml.write_text(
        "\n".join(
            [
                f"path: {output_dir}",
                "train: images/train",
                "val: images/val",
                "names:",
                names,
                "",
            ]
        )
    )

    print("Detector dataset prepared")
    print(f"dataset_yaml={dataset_yaml}")
    print(f"skipped_no_finding_rows={skipped_no_finding_rows}")
    print(dict(sorted(counts.items())))
    return dataset_yaml

def load_image_sizes(input_dir: Path) -> dict[str, tuple[float, float]]:
    """Load image_id -> (height, width) from img_size.csv."""

    return {
        Path(row["image_id"]).stem: (float(row["dim0"]), float(row["dim1"]))
        for row in read_csv_dicts(input_dir / "img_size.csv")
    }


def format_number(value: float) -> str:
    """Format prediction values compactly for Kaggle CSV output."""

    return f"{float(value):.6f}".rstrip("0").rstrip(".")


def prediction_string_from_result(result, *, target_height: float, target_width: float) -> str:
    """Convert one Ultralytics result into a scaled Kaggle PredictionString."""

    detections = []
    boxes = getattr(result, "boxes", None)
    if boxes is not None and len(boxes) > 0:
        source_height, source_width = getattr(result, "orig_shape", (target_height, target_width))
        x_scale = target_width / float(source_width)
        y_scale = target_height / float(source_height)
        xyxy_rows = boxes.xyxy.detach().cpu().tolist()
        confidence_rows = boxes.conf.detach().cpu().tolist()
        class_rows = boxes.cls.detach().cpu().tolist()
        for box, score, class_id in zip(xyxy_rows, confidence_rows, class_rows):
            class_id = int(class_id)
            if not 0 <= class_id <= 13:
                continue
            x_min, y_min, x_max, y_max = (float(value) for value in box)
            x_min = max(0.0, min(target_width, x_min * x_scale))
            x_max = max(0.0, min(target_width, x_max * x_scale))
            y_min = max(0.0, min(target_height, y_min * y_scale))
            y_max = max(0.0, min(target_height, y_max * y_scale))
            if x_min < x_max and y_min < y_max:
                detections.append((class_id, float(score), x_min, y_min, x_max, y_max))

    detections.sort(key=lambda detection: detection[1], reverse=True)
    detections = detections[:PRED_MAX_DETECTIONS]
    if not detections:
        return "14 1.0 0 0 1 1"

    tokens = []
    for class_id, score, x_min, y_min, x_max, y_max in detections:
        tokens.extend(
            [
                str(class_id),
                format_number(score),
                format_number(x_min),
                format_number(y_min),
                format_number(x_max),
                format_number(y_max),
            ]
        )
    return " ".join(tokens)


def batched(items: list[dict[str, str]], size: int):
    """Yield fixed-size chunks from a list."""

    if size <= 0:
        raise ValueError(f"Batch size must be positive, got {size}")
    for offset in range(0, len(items), size):
        yield items[offset : offset + size]


def write_submission(model, input_dir: Path, output_path: Path, *, device) -> dict[str, object]:
    """Run test inference and write a Kaggle submission CSV plus summary."""

    sample_rows = read_csv_dicts(input_dir / "sample_submission.csv")
    image_sizes = load_image_sizes(input_dir)
    test_images = input_dir / "test" / "test"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    counts = Counter()
    rows_written = 0

    predict_kwargs = {
        "imgsz": IMAGE_SIZE,
        "conf": PRED_CONF,
        "iou": PRED_IOU,
        "batch": PRED_BATCH,
        "max_det": PRED_MAX_DETECTIONS,
        "stream": True,
        "verbose": False,
    }
    if device != "cpu":
        predict_kwargs["device"] = device

    with output_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["image_id", "PredictionString"])
        writer.writeheader()
        for row_chunk in batched(sample_rows, PRED_CHUNK_SIZE):
            image_ids = [Path(str(row["image_id"])).stem for row in row_chunk]
            image_paths = []
            for image_id in image_ids:
                image_path = test_images / f"{image_id}.png"
                if not image_path.exists():
                    raise FileNotFoundError(f"Missing test image: {image_path}")
                image_paths.append(str(image_path))

            for row, image_id, result in zip(
                row_chunk,
                image_ids,
                model.predict(source=image_paths, **predict_kwargs),
            ):
                target_height, target_width = image_sizes[image_id]
                prediction_string = prediction_string_from_result(
                    result,
                    target_height=target_height,
                    target_width=target_width,
                )
                writer.writerow({"image_id": row["image_id"], "PredictionString": prediction_string})
                rows_written += 1
                tokens = prediction_string.split()
                for index in range(0, len(tokens), 6):
                    counts[tokens[index]] += 1
                if rows_written == 1 or rows_written % 250 == 0:
                    print(f"submission rows written {rows_written}/{len(sample_rows)}", flush=True)

    summary = {
        "output": str(output_path),
        "rows": rows_written,
        "expected_rows": len(sample_rows),
        "prediction_conf": PRED_CONF,
        "prediction_iou": PRED_IOU,
        "prediction_batch": PRED_BATCH,
        "prediction_chunk_size": PRED_CHUNK_SIZE,
        "prediction_max_detections": PRED_MAX_DETECTIONS,
        "detection_counts_by_class": dict(sorted(counts.items(), key=lambda item: int(item[0]))),
    }
    summary_path = output_path.with_name("submission-summary.json")
    summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
    print(f"Saved submission: {output_path}")
    print(f"Saved submission summary: {summary_path}")
    print(json.dumps(summary, indent=2, sort_keys=True))
    return summary

In [4]:
KAGGLE_INPUT = find_competition_input(KAGGLE_INPUT)
print(f"Kaggle input: {KAGGLE_INPUT}")
print(f"Working directory: {WORKING_DIR}")
print(
    "Training run: "
    f"kind={MODEL_KIND}, model={MODEL}, epochs={EPOCHS}, imgsz={IMAGE_SIZE}, "
    f"batch={BATCH}, workers={WORKERS}, patience={PATIENCE}, "
    f"train_time_hours={TRAIN_TIME_HOURS}, save_period={SAVE_PERIOD}, "
    f"deterministic={DETERMINISTIC}, run={RUN_NAME}"
)

ensure_ultralytics()
patch_ultralytics_ddp_find_unused_parameters()
dataset_yaml = prepare_detector_dataset(KAGGLE_INPUT, DETECTOR_DIR)

Kaggle input: /kaggle/input/competitions/amia-public-challenge-2026
Working directory: /kaggle/working
Training run: kind=rtdetr, model=rtdetr-l.pt, epochs=20, imgsz=640, batch=8, workers=4, patience=8, train_time_hours=10.5, save_period=5, deterministic=False, run=rtdetr-l-e20-b8-t4x2-ddpfix-t10h5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 18.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics=8.4.60
Patched Ultralytics DDP with find_unused_parameters=True: /usr/local/lib/python3.12/dist-packages/ultralytics/engine/trainer.py
Detector dataset prepared
dataset_yaml=/kaggle/working/amia-detector/dataset.yaml
ski

In [5]:
import shutil
from pathlib import Path

import torch
from ultralytics import RTDETR, YOLO


print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
print(f"cuda_device_count={torch.cuda.device_count()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        print(f"cuda_device_{index}={torch.cuda.get_device_name(index)}")

def parse_device(value: str):
    if value == "auto":
        return 0 if torch.cuda.is_available() else "cpu"
    if "," in value:
        return [int(item.strip()) for item in value.split(",") if item.strip()]
    return int(value) if value.isdigit() else value


def make_model(model_kind: str, model_name: str):
    if model_kind == "yolo":
        return YOLO(model_name)
    if model_kind == "rtdetr":
        return RTDETR(model_name)
    raise ValueError(f"Unsupported AMIA_MODEL_KIND={model_kind!r}; use yolo or rtdetr")


if DEVICE != "auto":
    train_device = parse_device(DEVICE)
elif torch.cuda.is_available():
    train_device = 0
else:
    train_device = "cpu"
print(f"train_device={train_device}")
if PRED_DEVICE != "auto":
    prediction_device = parse_device(PRED_DEVICE)
elif isinstance(train_device, list):
    prediction_device = train_device[0]
else:
    prediction_device = train_device
print(f"prediction_device={prediction_device}")

model = make_model(MODEL_KIND, MODEL)
train_kwargs = {
    "data": str(dataset_yaml),
    "epochs": EPOCHS,
    "imgsz": IMAGE_SIZE,
    "batch": BATCH,
    "workers": WORKERS,
    "project": str(MODEL_DIR),
    "name": RUN_NAME,
    "device": train_device,
    "exist_ok": True,
    "patience": PATIENCE,
    "save_period": SAVE_PERIOD,
    "cache": False,
    "deterministic": DETERMINISTIC,
}
if TRAIN_TIME_HOURS > 0:
    train_kwargs["time"] = TRAIN_TIME_HOURS
    print(f"Training will stop after at most {TRAIN_TIME_HOURS:g} hours, then generate outputs.")
else:
    print("Training has no explicit wall-clock limit; epochs/patience control completion.")

results = model.train(**train_kwargs)

save_dir = Path(results.save_dir if hasattr(results, "save_dir") else MODEL_DIR / RUN_NAME)
if not save_dir.exists():
    save_dir = MODEL_DIR / RUN_NAME
print(results)
print(f"Run directory: {save_dir}")

checkpoint_candidates = [
    save_dir / "weights" / "best.pt",
    save_dir / "weights" / "last.pt",
]
submission_weights = next((path for path in checkpoint_candidates if path.exists()), None)
if submission_weights is None:
    raise FileNotFoundError(
        "Training finished but no usable checkpoint was found. "
        f"Checked: {checkpoint_candidates}"
    )
optional_outputs = [save_dir / "args.yaml", save_dir / "results.csv"]
missing_optional_outputs = [path for path in optional_outputs if not path.exists()]
if missing_optional_outputs:
    print(f"Warning: optional training outputs are missing: {missing_optional_outputs}")
print(f"Generating submission from checkpoint: {submission_weights}")

submission_model = make_model(MODEL_KIND, str(submission_weights))
submission_path = save_dir / "submission.csv"
submission_summary = write_submission(
    submission_model,
    KAGGLE_INPUT,
    submission_path,
    device=prediction_device,
)
if submission_summary["rows"] != submission_summary["expected_rows"]:
    raise ValueError(f"Submission row mismatch: {submission_summary}")
working_submission = WORKING_DIR / f"{RUN_NAME}-submission.csv"
shutil.copy2(submission_path, working_submission)
print(f"Copied standalone submission: {working_submission}")

archive_base = WORKING_DIR / RUN_NAME
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=save_dir.parent,
    base_dir=save_dir.name,
)
print(f"Packaged artifacts: {archive_path}")
print("Download this zip from Kaggle output or commit the notebook output before ending the session.")

torch=2.10.0+cu128
cuda_available=True
cuda_device_count=2
cuda_device_0=Tesla T4
cuda_device_1=Tesla T4
train_device=[0, 1]
prediction_device=0
Training will stop after at most 10.5 hours, then generate outputs.
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/amia-detector/dataset.yaml, degrees=0.0, deterministic=False, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz